In [ ]:
import scanpy as sc

In [ ]:
adata_donor = sc.read_h5ad('/ix/djishnu/peasena/primary_multiome/donor1_upmc/h5_files/donor1_multiome_gex_post_mira.h5ad')

# Check aggregated file
adata_donor

In [ ]:
adata_tonsil = sc.read_h5ad('/ix/cigcore/sbg57/multiome/tonsil_atlas/seurat_object.h5ad')
# we convert 'annotation_figure_1' to 'Classifcation' in steps below

# Check aggregated file
adata_tonsil

In [ ]:
# processed object already has mapping tags to transfer Classification
adata_tonsil_1 = sc.read_h5ad('/ix/cigcore/sbg57/multiome/tonsil_atlas/bc_tonsil_processed.h5ad')

# Check aggregated file
adata_tonsil_1.obs.columns

In [ ]:
adata_tonsil_1.obs['Classification'].unique()

In [ ]:
# Add a new column "Classification" and extract everything after the last '_'
adata_tonsil_1.obs['Tag'] = adata_tonsil_1.obs.index.str.split('_').str[-1]

In [ ]:
# use cell index tags/batch to transfer the Classfication column from seurat object
# Ensure required columns exist in both datasets
if all(col in adata_tonsil_1.obs.columns for col in ['Tag', 'batch']) and \
   all(col in adata_tonsil.obs.columns for col in ['Tag', 'donor_id', 'annotation_figure_1']):
    
    # Create a mapping for rows in adata_invivo that match the donor_id condition
    annotation_map = adata_tonsil_1.obs.loc[
        adata_tonsil_1.obs['donor_id'] == 'BCLL-8-T', ['Tag', 'annotation_figure_1']
    ].set_index('Tag')['annotation_figure_1'].to_dict()

    # Apply mapping to the Classification column in adata
    adata_tonsil.obs['Classification'] = adata_tonsil.obs.apply(
        lambda row: annotation_map[row['Tag']] 
        if row['batch'] == 'in_vivo_male_child_1a' and row['Tag'] in annotation_map 
        else row.get('Classification', None), 
        axis=1
    )

    # Display rows updated for validation
    print(adata_tonsil_1.obs.loc[adata_tonsil_1.obs['Classification'].notna()])
else:
    print("Required columns are missing in one of the objects.")

In [ ]:
# Assuming adata_invitro and adata_invivo are your two datasets
sc.pp.highly_variable_genes(adata_donor)
sc.pp.highly_variable_genes(adata_tonsil_1)

# Use only highly variable genes in both datasets
adata_donor = adata_donor[:, adata_donor.var.highly_variable]
adata_tonsil_1 = adata_tonsil_1[:, adata_tonsil_1.var.highly_variable]

In [ ]:
# Get the common genes between the two datasets
common_genes = adata_donor.var_names.intersection(adata_tonsil_1.var_names)

# Subset both datasets to have the same features
adata_donor = adata_donor[:, common_genes]
adata_tonsil_1 = adata_tonsil_1[:, common_genes]
adata_tonsil = adata_tonsil_1

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Extract data and labels from AnnData objects
X_train = adata_tonsil.X.toarray() if not isinstance(adata_tonsil.X, np.ndarray) else adata_tonsil.X
y_train = adata_tonsil.obs["Classification"]  # Replace with the correct label column
X_test = adata_donor.X.toarray() if not isinstance(adata_donor.X, np.ndarray) else adata_donor.X

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM
svm = SVC(kernel="linear", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

# Predict probabilities on the test set
probabilities = svm.predict_proba(X_test_scaled)

# Set a threshold
threshold = 0.01  # Confidence level
predicted_labels = svm.predict(X_test_scaled)
max_probabilities = probabilities.max(axis=1)

# Replace low-confidence predictions with "NA"
final_predictions = [
    label if max_prob >= threshold else "NA"
    for label, max_prob in zip(predicted_labels, max_probabilities)
]

# Convert probabilities to a pandas DataFrame
probabilities_df = pd.DataFrame(
    probabilities,
    columns=[f"prob_{cls}" for cls in svm.classes_],  # Column names based on classes
    index=adata_donor.obs.index  # Ensure alignment with AnnData index
)

# Add the final predictions to the adata_donor.obs
adata_donor.obs["predicted_labels"] = final_predictions
adata_donor.obs = pd.concat([adata_donor.obs, probabilities_df], axis=1)

# Visualize or analyze the predictions
print(adata_donor.obs[["leiden", "predicted_labels"]].head())

In [ ]:
import scanpy as sc

# Visualize the mapping of in vitro cells with predicted labels
sc.pl.umap(adata_donor, color='prob_GCBC')
sc.pl.umap(adata_donor, color='prob_PC')
